# Chapter 17 &mdash; BDD Sizes, Dynamic Reordering, and the NP-Completeness of Ordering

**Concept 7 of the Chapter 17 decomposition:** *BDD Sizes, Dynamic Reordering, and the NP-Completeness of Ordering*

Good orders often give polynomial BDDs; sizes blow up during manipulation, and finding the best order is NP-complete.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-BDD-Sizes-And-Reordering/Concept-BDD-Sizes-And-Reordering.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Three facts you need before using BDDs in anger.

* **Good orders often exist.** Circuits with locality &mdash; adders, comparators,
  control logic &mdash; have polynomial BDDs under a sensible order, usually one that
  follows the circuit's own structure.
* **Sizes blow up during manipulation, not at the end.** The final result can be tiny
  while an intermediate is enormous. This is the practical failure mode, and it is why
  packages monitor node counts and reorder **dynamically** (sifting).
* **Finding the optimal order is NP-complete**, and even improving on a given order is
  hard. So packages use heuristics, not optimisation.

And a hard limit: integer **multiplication** has exponential BDDs under *every*
order. No heuristic rescues it.

## 2. Definitions

### The BDD package

In [ ]:
# --- a minimal BDD package ----------------------------------------------
# A node is either the terminal 0/1, or ('n', var_index, low, high) where
# low is the 0-branch and high the 1-branch.  Hash consing (the `unique`
# table) is what makes the representation canonical: structurally equal
# subgraphs become the SAME Python object, so equality is pointer equality.
ZERO, ONE = 0, 1

class BDD:
    def __init__(self, nvars):
        self.nvars = nvars
        self.unique = {}          # (var, low, high) -> node  -- hash consing
        self.apply_cache = {}

    def mk(self, var, low, high):
        if low is high: return low            # REDUCTION 1: skip a useless test
        key = (var, id(low), id(high), self._k(low), self._k(high))
        if key in self.unique: return self.unique[key]   # REDUCTION 2: share
        node = ('n', var, low, high)
        self.unique[key] = node
        return node

    def _k(self, n):
        return n if n in (ZERO, ONE) else ('n', n[1], self._k(n[2]), self._k(n[3]))

    def var(self, i):
        return self.mk(i, ZERO, ONE)

    def apply(self, op, a, b):
        key = (op, self._k(a), self._k(b))
        if key in self.apply_cache: return self.apply_cache[key]
        if a in (ZERO, ONE) and b in (ZERO, ONE):
            r = ONE if op(bool(a), bool(b)) else ZERO
        else:
            va = a[1] if a not in (ZERO, ONE) else self.nvars
            vb = b[1] if b not in (ZERO, ONE) else self.nvars
            v = min(va, vb)
            al, ah = (a[2], a[3]) if va == v else (a, a)
            bl, bh = (b[2], b[3]) if vb == v else (b, b)
            r = self.mk(v, self.apply(op, al, bl), self.apply(op, ah, bh))
        self.apply_cache[key] = r
        return r

    def NOT(self, a):  return self.apply(lambda x, y: not x, a, a)
    def AND(self, a, b): return self.apply(lambda x, y: x and y, a, b)
    def OR(self, a, b):  return self.apply(lambda x, y: x or y, a, b)
    def XOR(self, a, b): return self.apply(lambda x, y: x != y, a, b)

    def evaluate(self, node, assign):
        while node not in (ZERO, ONE):
            node = node[3] if assign[node[1]] else node[2]
        return bool(node)

    def size(self, node):
        seen = set()
        def walk(n):
            if n in (ZERO, ONE): return
            k = self._k(n)
            if k in seen: return
            seen.add(k); walk(n[2]); walk(n[3])
        walk(node)
        return len(seen)

    def onset(self, node, order=None):
        from itertools import product
        out = []
        for bits in product([False, True], repeat=self.nvars):
            a = {i: bits[i] for i in range(self.nvars)}
            if self.evaluate(node, a):
                out.append(''.join('1' if bits[i] else '0' for i in range(self.nvars)))
        return sorted(out)

### Building the same function under different orders

In [ ]:
def build_under(order, N, spec):
    # order: a permutation giving each logical variable its BDD level
    b = BDD(N)
    v = {logical: b.var(level) for logical, level in enumerate(order)}
    return b, spec(b, v)

def comparator(b, v):
    n = len(v) // 2
    gt, eq = ZERO, ONE
    for k in range(n):
        x, y = v[k], v[n + k]
        gt = b.OR(gt, b.AND(eq, b.AND(x, b.NOT(y))))
        eq = b.AND(eq, b.NOT(b.XOR(x, y)))
    return gt

<!-- nav-strip -->

---

&larr;&nbsp;[Ch17&nbsp;6.&nbsp;Canonicity via Myhill–Nerode, Hash Consing, and the Apply Operation](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Canonicity-And-Apply/Concept-Canonicity-And-Apply.ipynb) &nbsp;&middot;&nbsp; [**Chapter 17** index](https://github.com/ganeshutah/Jove/blob/master/Chapter17/README.md) &nbsp;&middot;&nbsp; [Ch18&nbsp;1.&nbsp;The History of Lambda Calculus, and its Independence from Turing's Work](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-History-Of-Lambda/Concept-History-Of-Lambda.ipynb)&nbsp;&rarr;

---

## 3. Tests

Order changes size dramatically, for the same function.

In [ ]:
import itertools
n = 3
N = 2 * n
inter = [0, 2, 4, 1, 3, 5]          # logical x0,x1,x2,y0,y1,y2 -> interleaved levels
separ = [0, 1, 2, 3, 4, 5]
for name, order in [('interleaved', inter), ('separated', separ)]:
    b, g = build_under(order, N, comparator)
    print("  %-12s : %d nodes" % (name, b.size(g)))
b1, g1 = build_under(inter, N, comparator)
b2, g2 = build_under(separ, N, comparator)
assert b1.size(g1) < b2.size(g2)

Searching all orders shows how wide the spread is.

In [ ]:
best, worst = None, None
for perm in itertools.permutations(range(6)):
    b, g = build_under(list(perm), 6, comparator)
    s = b.size(g)
    if best is None or s < best[0]: best = (s, perm)
    if worst is None or s > worst[0]: worst = (s, perm)
print("best  : %d nodes with order %s" % best)
print("worst : %d nodes with order %s" % worst)
print("spread: %.1fx" % (worst[0] / float(best[0])))
assert worst[0] > best[0]

**Blow-up happens in the middle**, not at the end.

In [ ]:
N = 8
b = BDD(N)
xs = [b.var(i) for i in range(N)]
peak, sizes = 0, []
g = xs[0]
for i, x in enumerate(xs[1:], 1):
    g = b.XOR(g, x)
    sizes.append(b.size(g))
    peak = max(peak, len(b.unique))
print("final BDD nodes :", b.size(g))
print("sizes along the way :", sizes)
print("unique-table peak   :", peak)
print("\nHere the growth is gentle.  For real circuits the intermediate")
print("can be orders of magnitude larger than the answer.")

Finding the **optimal** order is NP-complete, so packages use heuristics.

In [ ]:
from math import factorial
print("%-6s %s" % ("vars", "orders to try for an exhaustive search"))
for n in [6, 10, 16, 24]:
    print("%-6d %s" % (n, format(factorial(n), ',')))
print()
HEURISTICS = ["sifting: move one variable through all levels, keep the best",
              "window permutation: optimise k adjacent levels at a time",
              "structural: follow the circuit's topological order",
              "interleave related bit-vectors (as in the comparator)"]
for h in HEURISTICS: print("  *", h)

A crude sifting pass, to show the shape of the heuristic.

In [ ]:
def sift(N, spec, rounds=2):
    order = list(range(N))
    b, g = build_under(order, N, spec)
    best = b.size(g)
    for _ in range(rounds):
        for i in range(N):
            for j in range(N):
                if i == j: continue
                trial = order[:]
                trial[i], trial[j] = trial[j], trial[i]
                b2, g2 = build_under(trial, N, spec)
                if b2.size(g2) < best:
                    best, order = b2.size(g2), trial
    return best, order

start_b, start_g = build_under(list(range(6)), 6, comparator)
print("starting order : %d nodes" % start_b.size(start_g))
s, o = sift(6, comparator)
print("after sifting  : %d nodes with order %s" % (s, o))
assert s <= start_b.size(start_g)

And the hard limit.

In [ ]:
print("integer multiplication : exponential BDDs under EVERY variable order")
print("   (Bryant 1991 -- the middle output bit needs 2^(n/8) nodes)")
print()
print("So: BDDs are excellent for control logic and comparisons, and the")
print("wrong tool for arithmetic data paths.  Knowing which is which is")
print("the practical skill.")

## 4. Exercises


1. Run `sift` on a function of your own. How much does it help?
2. Why does dynamic reordering have to be triggered by node-count growth?
3. What representations are used instead of BDDs for multipliers?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter17/Concept-BDD-Sizes-And-Reordering')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')